# The paper's results, replayed

Every figure and table in the paper, rebuilt from the recorded results
rather than recomputed. Nothing here trains anything; the whole
notebook runs in seconds against the published data.

If a cell below refuses, it is because the data it needs has not been
fetched. The refusal names the command that fixes it.


In [ ]:
from IPython.display import Markdown, display

from mbl.replay import (
    GateFailedError,
    StoreIncompleteError,
    load_study,
    render_figure,
    render_composed_figure,
    render_gates,
    render_problem,
    render_protocol,
    render_provenance,
    resolve,
)

TIER = "publication_b16k"
STRESS_TIER = "publication"

depth = load_study("icassp_exact_convex/fig1_depth", tier=TIER)
display(Markdown(render_problem(depth)))


## What is being compared, and how

Every controller is scored on the same plant, from the same initial
states, under the same noise, at the same numerical precision. The
comparison is fair by construction rather than by convention: the
quantities below are inference costs on held-out trajectories, never
training losses.

The convex policy here solves its per-step program exactly rather than
approximately, and the bound beside it is computed from the same
program. Both are therefore properties of the problem being solved,
not of a solver's settings.


In [ ]:
display(Markdown(render_protocol(depth)))


## Figure 1 — cost against unfolding depth

The headline comparison. Depth is the number of unrolled iterations;
the flat series are the controllers that do not unroll and are drawn
at every depth for reference.


In [ ]:
try:
    figure_one = resolve(depth)
except (StoreIncompleteError, GateFailedError) as refusal:
    display(
        Markdown(
            "> **This result cannot be replayed.**\n>\n> ```text\n> "
            + str(refusal).replace("\n", "\n> ")
            + "\n> ```"
        )
    )
    raise

display(Markdown(render_gates(figure_one)))


In [ ]:
display(
    Markdown(
        render_figure(
            figure_one,
            "fig1_cost_vs_depth_exact",
            claim=(
                "Attained cost against unfolding depth, against the "
                "non-unrolling references and the exact convex bound."
            ),
        )
    )
)


The table behind the figure. Every row is one contender at one depth:
the aggregate over seeds, the spread within a seed and across seeds,
and how many trajectories each number rests on.


In [ ]:
figure_one.analyses["cost_by_depth_exact"].table


## Figure 2 — cost against mismatch severity

Three conditions, swept over the same severity axis: the controller is
told nothing about the mismatch, told about it, or fitted in the world
that produces it.

**This figure is assembled from three separate measurements**, one per
condition, and no single one of them holds it. Its address is derived
from the three below, so what is shown is the figure built from exactly
these measurements and not one that merely resembles it. The three
tables it is drawn from follow it.


In [ ]:
conditions = {}
for condition in ("blind", "told", "world"):
    study = load_study(
        f"icassp_exact_convex/fig2_angle_{condition}", tier=TIER
    )
    try:
        conditions[condition] = resolve(study)
    except (StoreIncompleteError, GateFailedError) as refusal:
        display(
            Markdown(
                "> **This result cannot be replayed.**\n>\n> ```text\n> "
                + str(refusal).replace("\n", "\n> ")
                + "\n> ```"
            )
        )
        raise

conditions["blind"].analyses["cost_by_angle_blind_exact"].table


In [ ]:
display(
    Markdown(
        render_composed_figure(
            [conditions[c] for c in ("blind", "told", "world")],
            "fig2_mismatch_severity_exact",
            title="Cost against mismatch severity",
            claim=(
                "Attained cost against mismatch severity, in the three "
                "conditions the paper compares."
            ),
        )
    )
)


In [ ]:
conditions["told"].analyses["cost_by_angle_told_exact"].table


In [ ]:
conditions["world"].analyses["cost_by_angle_world_exact"].table


## Figure 3 — a box that binds

The same depth comparison on a much larger plant, and under a control
bound tight enough that most control entries sit against it. Where the
first figure's constraint is mostly slack, here it is the dominant
feature of the problem, and the ordering it produces is the paper's
stress test rather than a repetition of the first result.

The learning rate differs from the one the first two figures use. That
is measured, not inherited: at this bound the rate that suits the
looser problem is unstable across seeds, and the rate below was chosen
by sweeping it with every other setting held fixed.

**The study below is called `fig5`.** That is our experiment numbering
and it is deliberately left alone: every stored record is named from
the document that declares it, so renaming the document to match the
paper would detach the results this figure is drawn from.


In [ ]:
stress = load_study(
    "icassp_exact_convex/fig5_stress_depth", tier=STRESS_TIER
)
try:
    figure_three = resolve(stress)
except (StoreIncompleteError, GateFailedError) as refusal:
    display(
        Markdown(
            "> **This result cannot be replayed.**\n>\n> ```text\n> "
            + str(refusal).replace("\n", "\n> ")
            + "\n> ```"
        )
    )
    raise

display(
    Markdown(
        render_figure(
            figure_three,
            "fig5_stress_cost_vs_depth",
            claim=(
                "Attained cost against unfolding depth at the binding "
                "bound, on the large plant."
            ),
        )
    )
)


In [ ]:
figure_three.analyses["cost_by_depth_stress"].table


## Where these numbers came from

The block below records what produced every result above: the exact
state of the code, the settings each measurement was taken under, and
the identifiers that tie a number to the run that made it. Two are
shown because the two instances were measured separately.


In [ ]:
display(Markdown(render_provenance(figure_one)))


In [ ]:
display(Markdown(render_provenance(figure_three)))
